In [ ]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
""" Problème de livraisons """

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp


def create_data_model():
    """ Permet de créer les données du problème"""
    data = {}
    data["distance_matrix"] = [
      [0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536, 502, 388, 354, 468, 776, 662],
      [548, 0, 684, 308, 194, 502, 730, 354, 696, 742, 1084, 594, 480, 674, 1016, 868, 1210],
      [776, 684, 0, 992, 878, 502, 274, 810, 468, 742, 400, 1278, 1164, 1130, 788, 1552, 754],
      [696, 308, 992, 0, 114, 650, 878, 502, 844, 890, 1232, 514, 628, 822, 1164, 560, 1358],
      [582, 194, 878, 114, 0, 536, 764, 388, 730, 776, 1118, 400, 514, 708, 1050, 674, 1244],
      [274, 502, 502, 650, 536, 0, 228, 308, 194, 240, 582, 776, 662, 628, 514, 1050, 708],
      [502, 730, 274, 878, 764, 228, 0, 536, 194, 468, 354, 1004, 890, 856, 514, 1278, 480],
      [194, 354, 810, 502, 388, 308, 536, 0, 342, 388, 730, 468, 354, 320, 662, 742, 856],
      [308, 696, 468, 844, 730, 194, 194, 342, 0, 274, 388, 810, 696, 662, 320, 1084, 514],
      [194, 742, 742, 890, 776, 240, 468, 388, 274, 0, 342, 536, 422, 388, 274, 810, 468],
      [536, 1084, 400, 1232, 1118, 582, 354, 730, 388, 342, 0, 878, 764, 730, 388, 1152, 354],
      [502, 594, 1278, 514, 400, 776, 1004, 468, 810, 536, 878, 0, 114, 308, 650, 274, 844],
      [388, 480, 1164, 628, 514, 662, 890, 354, 696, 422, 764, 114, 0, 194, 536, 388, 730],
      [354, 674, 1130, 822, 708, 628, 856, 320, 662, 388, 730, 308, 194, 0, 342, 422, 536],
      [468, 1016, 788, 1164, 1050, 514, 514, 662, 320, 274, 388, 650, 536, 342, 0, 764, 194],
      [776, 868, 1552, 560, 674, 1050, 1278, 742, 1084, 810, 1152, 274, 388, 422, 764, 0, 798],
      [662, 1210, 754, 1358, 1244, 708, 480, 856, 514, 468, 354, 844, 730, 536, 194, 798, 0],
    ]
    data["demands"] = [0, 1, 1, 2, 4, 2, 4, 8, 8, 1, 2, 1, 2, 4, 4, 8, 8]
    data["vehicle_capacities"] = [20, 20, 20, 20]
    data["pickups_deliveries"] = [
        [1, 6],
        [2, 10],
        [4, 3],
        [5, 9],
        [7, 8],
        [15, 11],
        [13, 12],
        [16, 14],
    ]
    data["num_vehicles"] = 4
    data["depot"] = 0
    return data


def print_solution(data, manager, routing, solution):
    """ Affiche la solution """
    print(f"Valeur de la fonction objectif : {solution.ObjectiveValue()}\n")
    total_distance = 0
    total_load = 0
    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        plan_output = f"Route pour le véhicule {vehicle_id} :\n"
        route_distance = 0
        route_load = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_load += data["demands"][node_index]
            plan_output += f"Dépôt n°{node_index}, Charge({route_load}) -> "
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )
        plan_output += f"Dépôt n°{manager.IndexToNode(index)} Charge({route_load})\n"
        plan_output += f"Distance parcourue par le véhicule : {route_distance}m\n"
        plan_output += f"Charge totale du véhicule  : {route_load}\n"
        print(plan_output)
        total_distance += route_distance
        total_load += route_load
    print(f"Distance totale parcourue par tous les véhicules : {total_distance}m")
    print(f"Charge totale de tous les véhicules : {total_load}")


def main():
    # Crée les données du problème
    data = create_data_model()

    # Création du modèle permettant d'associer les indices aux dépôts.
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]), data["num_vehicles"], data["depot"]
    )

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        """Renvoie la distance entre deux dépôts selon leurs indices"""
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data["distance_matrix"][from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    # Définit le coût des arcs
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        """Renvoie la quantité nécessaire au dépôt qui correspond à l'indice"""
        from_node = manager.IndexToNode(from_index)
        return data["demands"][from_node]

    # Crée une distance permettant de calculer la charge totale
    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data["vehicle_capacities"],  # Charge maximum d'un véhicule
        True,
        "Capacity",
    )

    # Crée une distance permettant de calculer la distance parcourue
    routing.AddDimension(
        transit_callback_index,
        0,
        3000,  # Distance maximum que peux parcourir un véhicule
        True,
        "Distance",
    )
    distance_dimension = routing.GetDimensionOrDie("Distance")
    distance_dimension.SetGlobalSpanCostCoefficient(100)

    # Définit les demandes de transport
    for request in data["pickups_deliveries"]:
        pickup_index = manager.NodeToIndex(request[0])
        delivery_index = manager.NodeToIndex(request[1])
        routing.AddPickupAndDelivery(pickup_index, delivery_index)
        routing.solver().Add(
            routing.VehicleVar(pickup_index) == routing.VehicleVar(delivery_index)
        )
        routing.solver().Add(
            distance_dimension.CumulVar(pickup_index)
            <= distance_dimension.CumulVar(delivery_index)
        )

    # Résout le problème
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    solution = routing.SolveWithParameters(search_parameters)

    # Affiche la solution
    if solution:
        print_solution(data, manager, routing, solution)


if __name__ == "__main__":
    main()

Valeur de la fonction objectif : 230424

Route pour le véhicule 0 :
Dépôt n°0, Charge(0) -> Dépôt n°7, Charge(8) -> Dépôt n°8, Charge(16) -> Dépôt n°0 Charge(16)
Distance parcourue par le véhicule : 844m
Charge totale du véhicule  : 16

Route pour le véhicule 1 :
Dépôt n°0, Charge(0) -> Dépôt n°4, Charge(4) -> Dépôt n°3, Charge(6) -> Dépôt n°1, Charge(7) -> Dépôt n°6, Charge(11) -> Dépôt n°0 Charge(11)
Distance parcourue par le véhicule : 2236m
Charge totale du véhicule  : 11

Route pour le véhicule 2 :
Dépôt n°0, Charge(0) -> Dépôt n°5, Charge(2) -> Dépôt n°2, Charge(3) -> Dépôt n°10, Charge(5) -> Dépôt n°16, Charge(13) -> Dépôt n°14, Charge(17) -> Dépôt n°9, Charge(18) -> Dépôt n°0 Charge(18)
Distance parcourue par le véhicule : 2192m
Charge totale du véhicule  : 18

Route pour le véhicule 3 :
Dépôt n°0, Charge(0) -> Dépôt n°13, Charge(4) -> Dépôt n°15, Charge(12) -> Dépôt n°11, Charge(13) -> Dépôt n°12, Charge(15) -> Dépôt n°0 Charge(15)
Distance parcourue par le véhicule : 1552m
Ch

In [ ]:
"""Problème de plannification des infirmières"""
from typing import Union
from ortools.sat.python import cp_model


def main():
    # Ce programme cherche à déterminer une affectation optimale pour les infirmières
    # 3 temps de travail par jour, pendant 7 jours.
    # L'affectation optimale maximise le nombre de demandes satisfaites.
    num_nurses = 5
    num_days = 7
    num_shifts = 3

    all_nurses = range(num_nurses)
    all_days = range(num_days)
    all_shifts = range(num_shifts)

    shift_requests = [
        [[0, 0, 1], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 1], [0, 1, 0], [0, 0, 1]],
        [[0, 0, 0], [0, 0, 0], [0, 1, 0], [0, 1, 0], [1, 0, 0], [0, 0, 0], [0, 0, 1]],
        [[0, 1, 0], [0, 1, 0], [0, 0, 0], [1, 0, 0], [0, 0, 0], [0, 1, 0], [0, 0, 0]],
        [[0, 0, 1], [0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 0], [1, 0, 0], [0, 0, 0]],
        [[0, 0, 0], [0, 0, 1], [0, 1, 0], [0, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 0]],
    ]


    model = cp_model.CpModel()
    if not model:
      print("Erreur")

    # Création de variables entières
    shifts = {}
    for n in all_nurses:
        for d in all_days:
            for s in all_shifts:
                shifts[(n,d,s)] = model.new_bool_var(f"shift[({n},{d},{s})]")

    # Pas plus d'une infirmière en même temps
    for d in all_days:
        for s in all_shifts:
            model.add(sum(shifts[n0,d,s] for n0 in all_nurses) == 1)

    # Chaque infirmière doit effectuer au maximum un shift par jour
    for n in all_nurses:
        for d in all_days:
            model.add(sum(shifts[n,d,s0] for s0 in all_shifts) <= 1)

    # Il y a au maximum 3 shifts par jour
    for d in all_days:
        model.add(sum(shifts[n0,d,s0]*shift_requests[n0][d][s0] for n0 in all_nurses for s0 in all_shifts) <= 3)


    # Chaque infirmière à peu près le même nombre de shifts que les autre
    min_shifts_per_nurse = (num_shifts * num_days) // num_nurses
    if (num_shifts * num_days) % num_nurses == 0:
        max_shifts_per_nurse = min_shifts_per_nurse
    else:
        max_shifts_per_nurse = min_shifts_per_nurse + 1

    for n in all_nurses:
        model.add(sum(shifts[n,d0,s0] for d0 in all_days for s0 in all_shifts) <= max_shifts_per_nurse)
        model.add(sum(shifts[n,d0,s0] for d0 in all_days for s0 in all_shifts) >= min_shifts_per_nurse)

    model.Maximize(sum(shifts[(n0,d0,s0)]*shift_requests[n0][d0][s0] for n0 in all_nurses for d0 in all_days for s0 in all_shifts))

    print(f"{min_shifts_per_nurse}")


    solver = cp_model.CpSolver()
    status = solver.solve(model)

    if status == cp_model.OPTIMAL:
        print("Solution :")
        for d in all_days:
            print("Jour", d)
            for n in all_nurses:
                for s in all_shifts:
                    if solver.value(shifts[(n, d, s)]) == 1:
                        if shift_requests[n][d][s] == 1:
                            print("Infirmière", n, ", heures de travail", s, "(satisfait).")
                        else:
                            print("Infirmière", n, ", heures de travail", s, "(non satisfait).")
            print()
        print(
            f"Nombres de demande satisfaisantes = {solver.objective_value}",
            f"(sur {num_nurses * min_shifts_per_nurse})",
        )
    else:
        print("Pas de solution optimale !")


if __name__ == "__main__":
    main()

4
Solution :
Jour 0
Infirmière 0 , heures de travail 2 (satisfait).
Infirmière 2 , heures de travail 1 (satisfait).
Infirmière 4 , heures de travail 0 (non satisfait).

Jour 1
Infirmière 1 , heures de travail 0 (non satisfait).
Infirmière 2 , heures de travail 1 (satisfait).
Infirmière 4 , heures de travail 2 (satisfait).

Jour 2
Infirmière 1 , heures de travail 1 (satisfait).
Infirmière 3 , heures de travail 0 (satisfait).
Infirmière 4 , heures de travail 2 (non satisfait).

Jour 3
Infirmière 1 , heures de travail 1 (satisfait).
Infirmière 2 , heures de travail 0 (satisfait).
Infirmière 3 , heures de travail 2 (non satisfait).

Jour 4
Infirmière 0 , heures de travail 2 (satisfait).
Infirmière 1 , heures de travail 1 (non satisfait).
Infirmière 4 , heures de travail 0 (satisfait).

Jour 5
Infirmière 0 , heures de travail 1 (satisfait).
Infirmière 2 , heures de travail 2 (non satisfait).
Infirmière 3 , heures de travail 0 (satisfait).

Jour 6
Infirmière 0 , heures de travail 2 (satisfai

In [ ]:
""" Problème d'affectation de tâches à des machines """
from ortools.linear_solver import pywraplp

costs = [
    [90, 80, 75, 70],
    [35, 85, 55, 65],
    [125, 95, 90, 95],
    [45, 110, 95, 115],
    [50, 100, 90, 100],
]
num_workers = len(costs)
num_tasks = len(costs[0])


solver = pywraplp.Solver.CreateSolver("SCIP")

if not solver:
    print("Erreur")

# Création de variables
taches = {}
for w in range(num_workers):
  for t in range(num_tasks):
    taches[(w,t)] = solver.IntVar(0,1, f"taches_w{w}_t{t}")

# Chaque machine a au plus une tâche
for w in range(num_workers):
  solver.Add(sum(taches[(w,t)] for t in range(num_tasks)) <= 1)

# Exactement une tâche par machine
for t in range(num_tasks):
  solver.Add(sum(taches[(w,t)] for w in range(num_workers)) == 1)

solver.Minimize(sum(taches[(w,t)]*costs[w][t] for w in range(num_workers) for t in range(num_tasks)))


# Solution
print(f"Problème résolue avec {solver.SolverVersion()}\n")
status = solver.Solve()
print(f"Status : {status}")
if status == pywraplp.Solver.OPTIMAL:
    print(f"Solution de valeur : {solver.Objective().Value()}")
    for w in range(num_workers):
      for t in range(num_tasks):
        if(taches[(w,t)].solution_value()):
          print(f"La tache {t} est affecté à la machine {w}")

Problème résolue avec SCIP 10.0.0 [LP solver: SoPlex 8.0.0]

Status : 0
Solution de valeur : 265.0
La tache 3 est affecté à la machine 0
La tache 2 est affecté à la machine 1
La tache 1 est affecté à la machine 2
La tache 0 est affecté à la machine 3


In [ ]:
""" Problème multi sac-à-dos """
from ortools.linear_solver import pywraplp
data = {}
data["weights"] = [48, 30, 42, 36, 36, 48, 42, 42, 36, 24, 30, 30, 42, 36, 36]
data["values"] = [10, 30, 25, 50, 35, 30, 15, 40, 30, 35, 45, 10, 20, 30, 25]
assert len(data["weights"]) == len(data["values"])
data["num_items"] = len(data["weights"])
data["all_items"] = range(data["num_items"])

data["bin_capacities"] = [100, 100, 100, 100, 100]
data["num_bins"] = len(data["bin_capacities"])
data["all_bins"] = range(data["num_bins"])


solver = pywraplp.Solver.CreateSolver("SCIP")
if solver is None:
    print("SCIP solver unavailable.")

# Création de variables
x = {}
for item in data["all_items"]:
  for bin in data["all_bins"]:
    x[item, bin] = solver.IntVar(0, 1, f"x_item{item}_bin{bin}")

# Chaque objet est au plus dans une seule bin à la fois
for item in data["all_items"]:
  solver.Add(sum(x[item, bin] for bin in data["all_bins"]) <= 1)

# On ne peut pas dépasser le volume de la bin
for bin in data["all_bins"]:
  solver.Add(sum(x[item, bin]*data["weights"][item] for item in data["all_items"]) <= data["bin_capacities"][bin])


solver.Maximize(sum(x[item, bin]*data["values"][item] for item in data["all_items"] for bin in data["all_bins"]))



# Solution
print(f"Problème résolue avec {solver.SolverVersion()}\n")
status = solver.Solve()
print(f"Status : {status}")
if status == pywraplp.Solver.OPTIMAL:
    print(f"Valeur de la solution : {solver.Objective().Value()}")
    for bin in data["all_bins"]:
      print(f"\nBin n°{bin}")
      total_bin_weight = 0
      total_bin_value = 0
      for item in data["all_items"]:
        if(x[item, bin].solution_value()):
          print(f"L'objet {item} de poids {data["weights"][item]} et de valeur {data["values"][item]}")
          total_bin_weight += data["weights"][item]
          total_bin_value += data["values"][item]
      print(f"La valeur totale de la bin n°{bin} est {total_bin_value} de poids {total_bin_weight}")

Problème résolue avec SCIP 10.0.0 [LP solver: SoPlex 8.0.0]

Status : 0
Valeur de la solution : 395.0

Bin n°0
L'objet 4 de poids 36 et de valeur 35
L'objet 8 de poids 36 et de valeur 30
La valeur totale de la bin n°0 est 65 de poids 72

Bin n°1
L'objet 5 de poids 48 et de valeur 30
L'objet 7 de poids 42 et de valeur 40
La valeur totale de la bin n°1 est 70 de poids 90

Bin n°2
L'objet 2 de poids 42 et de valeur 25
L'objet 12 de poids 42 et de valeur 20
La valeur totale de la bin n°2 est 45 de poids 84

Bin n°3
L'objet 3 de poids 36 et de valeur 50
L'objet 9 de poids 24 et de valeur 35
L'objet 13 de poids 36 et de valeur 30
La valeur totale de la bin n°3 est 115 de poids 96

Bin n°4
L'objet 1 de poids 30 et de valeur 30
L'objet 10 de poids 30 et de valeur 45
L'objet 14 de poids 36 et de valeur 25
La valeur totale de la bin n°4 est 100 de poids 96
